In [1]:
import fastf1
import pandas as pd

In [2]:
fastf1.Cache.enable_cache("../data/cache") 
# Cache directory for storing downloaded data
# Useful because we do not have to download data over and over again

## Initial data loading 

In [3]:
session = fastf1.get_session(2023, "Bahrain", "R")
session.load()
# loading the race session, pretty basic

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [4]:
results = session.results
results.head() 

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:33:56.736000,Finished,25.0,57.0
11,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,...,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:11.987000,Finished,18.0,57.0
14,14,F ALONSO,ALO,alonso,Aston Martin,358C75,aston_martin,Fernando,Alonso,Fernando Alonso,...,3.0,3,5.0,NaT,NaT,NaT,0 days 00:00:38.637000,Finished,15.0,57.0
55,55,C SAINZ,SAI,sainz,Ferrari,F91536,ferrari,Carlos,Sainz,Carlos Sainz,...,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:48.052000,Finished,12.0,57.0
44,44,L HAMILTON,HAM,hamilton,Mercedes,6CD3BF,mercedes,Lewis,Hamilton,Lewis Hamilton,...,5.0,5,7.0,NaT,NaT,NaT,0 days 00:00:50.977000,Finished,10.0,57.0


In [5]:
results[[
    "Abbreviation", 
    "FullName", 
    "TeamName", 
    "GridPosition", 
    "Position", 
    "Status"
    ]]
# grid position pretty vital info ngl

,Abbreviation,FullName,TeamName,GridPosition,Position,Status
1,VER,Max Verstappen,Red Bull Racing,1.0,1.0,Finished
11,PER,Sergio Perez,Red Bull Racing,2.0,2.0,Finished
14,ALO,Fernando Alonso,Aston Martin,5.0,3.0,Finished
55,SAI,Carlos Sainz,Ferrari,4.0,4.0,Finished
44,HAM,Lewis Hamilton,Mercedes,7.0,5.0,Finished
18,STR,Lance Stroll,Aston Martin,8.0,6.0,Finished
63,RUS,George Russell,Mercedes,6.0,7.0,Finished
77,BOT,Valtteri Bottas,Alfa Romeo,12.0,8.0,Finished
10,GAS,Pierre Gasly,Alpine,20.0,9.0,Finished
23,ALB,Alexander Albon,Williams,15.0,10.0,Finished


In [6]:
results.columns  # Just gives column names, not the data itself

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'],
      dtype='object')

In [7]:
# first mini insight
results[["FullName", "TeamName", "GridPosition", "Position"]].sort_values(by="GridPosition") 
# grid position is the starting position
# position is just the finishing one
# Can notice a slight correlation here and there.

,FullName,TeamName,GridPosition,Position
1,Max Verstappen,Red Bull Racing,1.0,1.0
11,Sergio Perez,Red Bull Racing,2.0,2.0
16,Charles Leclerc,Ferrari,3.0,19.0
55,Carlos Sainz,Ferrari,4.0,4.0
14,Fernando Alonso,Aston Martin,5.0,3.0
63,George Russell,Mercedes,6.0,7.0
44,Lewis Hamilton,Mercedes,7.0,5.0
18,Lance Stroll,Aston Martin,8.0,6.0
31,Esteban Ocon,Alpine,9.0,18.0
27,Nico Hulkenberg,Haas F1 Team,10.0,15.0


## Preprocessing

In [8]:
results = results[results["Status"] == "Finished"] # retrieving only the drivers who finished
df = results[["Abbreviation", "TeamName", "GridPosition", "Position"]].copy()

df.dtypes

Abbreviation     object
TeamName         object
GridPosition    float64
Position        float64
dtype: object

In [9]:
# we change the grid position and position to int
df["GridPosition"] = df["GridPosition"].astype(int)
df["Position"] = df["Position"].astype(int)

df.dtypes

Abbreviation    object
TeamName        object
GridPosition     int64
Position         int64
dtype: object

In [10]:
df = pd.get_dummies(df, columns=["TeamName"])
# this is the process of using hot encoding (learnt from the ML lectures)
# because we cannot use categorical data in our model, we have to convert it to numerical data

In [11]:
x = df.drop(columns=["Abbreviation","Position"])
y = df["Position"]
# x is the features, y is the target variable

In [12]:
x.head()
y.head()

# we remove abbreviation because it is not as useful
# we remove position because that is what we are trying to predict
# If we keep it in the current way then it will not make sense as that is what we are trying to predict
# we bascially get a fake prediction

1     1
11    2
14    3
55    4
44    5
Name: Position, dtype: int64

## TIME TO MAKE THE MODEL

In [13]:
from sklearn.ensemble import RandomForestRegressor

In [14]:
model = RandomForestRegressor()
model.fit(x,y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [15]:
predictions = model.predict(x)

# make the predictions from the Random Forest model

In [16]:
compare = pd.DataFrame({
    "Actual": y,
    "Predicted": predictions
})

compare.head()
# here we compare the predicted values with the actual y values that we have
# for now my goal is to rank the driver positions correctly
# not to predict their exact positions

,Actual,Predicted
1,1,1.64
11,2,1.96
14,3,3.77
55,4,3.86
44,5,5.47


## EVALUATING THE MODEL

- Since this model worked on the training data, it does not make it very good
- So to counter this we will train it in random data

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# we split the data into training and testing sets, 80% for training and 20% for testing
# from the lecture assignments

In [18]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [19]:
predictions = model.predict(X_test)

In [20]:
from sklearn.metrics import mean_absolute_error

mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)
# this gives us an idea of how far off our predictiions will be

Mean Absolute Error: 1.3999999999999997


In [21]:
compare = pd.DataFrame({
    "Actual": y_test, 
    "Predicted": predictions
    })
compare.head()

,Actual,Predicted
18,6,5.16
1,1,3.00
23,10,8.64


Since the MAE we got is 1.54 our predictions are not as bas as expected
This is doing predictions only based off of one race,
- So now we upgrade this model
- We use more races
- Add more features also because we currenly use only one

Furthermore, we have encoded the teamnames for each driver.
- It will show 1 if they belong to that team
- 0 otherwise

To run this whole model we use the features **grid position** and **teamname** which is encoded.
We use those two to predict the final position

## Now we build the multi race dataset

For our model, we will get all the races from 2023

We will train our model from the 2023 races and then use in it to predict the results of the 2024

Bismillah

In [29]:
schedule_2023 = fastf1.get_event_schedule(2023)
schedule_2023[["RoundNumber", "EventName", "Country"]]
# this is just to check the schedule of 2023 season
# we need this for the next cell

,RoundNumber,EventName,Country
0,0,Pre-Season Testing,Bahrain
1,1,Bahrain Grand Prix,Bahrain
2,2,Saudi Arabian Grand Prix,Saudi Arabia
3,3,Australian Grand Prix,Australia
4,4,Azerbaijan Grand Prix,Azerbaijan
5,5,Miami Grand Prix,United States
6,6,Monaco Grand Prix,Monaco
7,7,Spanish Grand Prix,Spain
8,8,Canadian Grand Prix,Canada
9,9,Austrian Grand Prix,Austria


In [31]:
all_races = []

for _, event in schedule_2023.iterrows():
    try:
        race_name = event["EventName"]

        session = fastf1.get_session(2023, race_name, "R")
        session.load()

        race_results = session.results[[
            "Abbreviation",  
            "TeamName", 
            "GridPosition", 
            "Position", 
            "Status"
        ]].copy()

        race_results["RoundNumber"] = event["RoundNumber"] # had to add this line so races are ordered 
        race_results["RaceName"] = race_name
        race_results["Year"] = 2023

        all_races.append(race_results)

        print(f"Successfully loaded {race_name}")
    
    except Exception as e:
        print(f"Error occurred while loading {race_name}: {e}")

events      WARNING 	Correcting user input 'Pre-Season Testing' to 'British Grand Prix'
core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '44', '81', '63', '11', '14', '23', '16', '55', '2', '77', '27', '18', '24', '22

Successfully loaded Pre-Season Testing


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data


Successfully loaded Bahrain Grand Prix


req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...

Successfully loaded São Paulo Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.001000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '31', '18', '55', '44', '63', '14', '81', '10', '23', '20', '3', '24', '2', '77', '22', '27', '4']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req 

Successfully loaded Las Vegas Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '11', '4', '81', '14', '22', '44', '18', '3', '31', '10', '23', '27', '2', '24', '55', '77', '20']


Successfully loaded Abu Dhabi Grand Prix


In [32]:
multi_race_df = pd.concat(all_races, ignore_index=True)
multi_race_df.shape

(460, 8)

In [33]:
multi_race_df.columns

Index(['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Status',
       'RoundNumber', 'RaceName', 'Year'],
      dtype='object')

ALL THE RACES OF 2023 HAVE BEEN LOADED AND NOW WE REPEAT THE PREPROCESSING PROCESS BY DOING THE SAME THINGS

In [34]:
multi_race_df = multi_race_df[multi_race_df["Status"] == "Finished"]
## remove the drivers who did not finish

df = multi_race_df[[
    "Abbreviation", 
    "TeamName", 
    "GridPosition", 
    "Position"
    ]].copy()

In [35]:
df["GridPosition"] = df["GridPosition"].astype(int)
df["Position"] = df["Position"].astype(int)

df.dtypes

Abbreviation    object
TeamName        object
GridPosition     int64
Position         int64
dtype: object

In [36]:
df = pd.get_dummies(df, columns=["TeamName"])

X = df.drop(columns=["Abbreviation", "Position"])
y = df["Position"]

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [39]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)

mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

# Here we can notice that MAE will be higher than before
# This is because we are now working with more data so there will be very big differences
# Currently we are only predicing the position based on the grid position and team name
# There are many other factors we should consider so we will put it into account later
# CURRENT MAE IS 2.6129...

Mean Absolute Error: 2.6282742905159195


## We begin feature engineering

In [40]:
multi_race_df = multi_race_df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])
# we first sort the data by the driver abbreviation, then by year and then by round number

In [41]:
multi_race_df["PreviousPosition"] = multi_race_df.groupby("Abbreviation")["Position"].shift(1)

# So this is very important
# We use this new feature that we made in order ot predict the position of the driver in the next race basically

multi_race_df[["Abbreviation", "Year", "RaceName", "Position", "PreviousPosition"]].head(10)
# Round number came in clutch and managed to order things accordingly

,Abbreviation,Year,RaceName,Position,PreviousPosition
7,ALB,2023,Pre-Season Testing,8.0,NaN
29,ALB,2023,Bahrain Grand Prix,10.0,8.0
91,ALB,2023,Azerbaijan Grand Prix,12.0,10.0
113,ALB,2023,Miami Grand Prix,14.0,12.0
166,ALB,2023,Canadian Grand Prix,7.0,14.0
190,ALB,2023,Austrian Grand Prix,11.0,7.0
207,ALB,2023,British Grand Prix,8.0,11.0
253,ALB,2023,Belgian Grand Prix,14.0,8.0
267,ALB,2023,Dutch Grand Prix,8.0,14.0
286,ALB,2023,Italian Grand Prix,7.0,8.0


In [42]:
multi_race_df = multi_race_df.dropna(subset=["PreviousPosition"])
# dropping the results if its not a number

In [43]:
df = multi_race_df[[ 
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition"
]]

df = pd.get_dummies(df, columns=["TeamName"]) # encoding
# FORGOT TO EXPLAIN HOW IT WORKS
# Since team name is text ML models cannot understand it.
# So we use one hot encoding to convert it into numerical data
# If we had a column of team names like Mercedes, Ferrari, Red Bull
# We would create 3 new columns called Team_Mercedes, Team_Ferrari, Team_RedBull
# Then it would put 1 in the column of the team that the driver is in and 0 in the other columns
# So if a driver is in Mercedes, it would put 1 in Team_Mercedes and 0 in Team_Ferrari and Team_RedBull

In [44]:
X = df.drop(columns=["Position"]) # drop positions but keep the rest
y = df["Position"]
# We remove position because that is what we are trying to predict, we cannot use it as a feature

# FOR MY UNDERSTANDING
# THE ABOVE DF USES 4 COLUMNS WHICH ARE GRID POSITION, PREVIOUS POSITION AND THE TEAM NAME ENCODED INTO NUMERICAL DATA
# SO THAT DF IS THE ONE THAT WE ARE PROVIDING TO OUR MODEL TO MAKE PREDICTIONS

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.0127218869093872


In [46]:
compare = pd.DataFrame({
    "Driver": multi_race_df.loc[X_test.index, "Abbreviation"],
    "Actual": y_test, 
    "Predicted": predictions
    })
compare.head(20)

,Driver,Actual,Predicted
340,VER,1.0,1.119818
20,VER,1.0,1.119818
263,PER,4.0,1.920000
310,ALB,11.0,11.640000
202,HAM,3.0,6.273333
280,VER,1.0,1.438884
271,HUL,12.0,13.311667
242,LEC,3.0,3.497333
207,ALB,8.0,11.880000
67,PIA,8.0,10.612500


## We do new feature engineering now: CHECKING AVERAGE POSITION OF 3 RACES

In [47]:
multi_race_df = multi_race_df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

In [48]:
multi_race_df["Rolling3Average"] = (
    multi_race_df.groupby("Abbreviation")["Position"]
    .rolling(3)
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)
# kinda makes sense what is going on ngl
# so we get rolling then the average and then shift it by 1

In [49]:
multi_race_df[
    multi_race_df["Abbreviation"] == "ALO"
    ][["RoundNumber", "Position", "PreviousPosition", "Rolling3Average"]]

,RoundNumber,Position,PreviousPosition,Rolling3Average
22,1,3.0,7.0,11.666667
42,2,3.0,3.0,NaN
62,3,3.0,3.0,NaN
83,4,4.0,3.0,3.000000
102,5,3.0,4.0,3.333333
121,6,2.0,3.0,3.333333
146,7,7.0,2.0,3.000000
161,8,2.0,7.0,4.000000
184,9,5.0,2.0,3.666667
206,10,7.0,5.0,4.666667


In [50]:
multi_race_df = multi_race_df.dropna(subset=["Rolling3Average"])

multi_race_df[
    multi_race_df["Abbreviation"] == "VER"
    ][["RoundNumber", "Position", "PreviousPosition", "Rolling3Average"]]

,RoundNumber,Position,PreviousPosition,Rolling3Average
20,1,1.0,1.0,9.666667
81,4,2.0,1.0,1.333333
100,5,1.0,2.0,1.666667
120,6,1.0,1.0,1.333333
140,7,1.0,1.0,1.333333
160,8,1.0,1.0,1.000000
180,9,1.0,1.0,1.000000
200,10,1.0,1.0,1.000000
220,11,1.0,1.0,1.000000
240,12,1.0,1.0,1.000000


In [51]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average"
]].copy()

In [52]:
df = pd.get_dummies(df, columns=["TeamName"])

In [53]:
X = df.drop(columns=["Position"])
y = df["Position"]

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 1.8166232492997196


In [55]:
compare = pd.DataFrame({
    "Driver": multi_race_df.loc[X_test.index, "Abbreviation"],
    "Actual": y_test, 
    "Predicted": predictions
    })
compare.head(20)

,Driver,Actual,Predicted
425,SAI,6.0,6.980000
310,ALB,11.0,11.920000
312,HUL,13.0,12.200000
270,STR,11.0,10.530000
444,NOR,5.0,4.830000
300,SAI,1.0,4.400000
168,STR,9.0,10.440000
284,RUS,5.0,7.000000
388,ALB,9.0,11.560000
27,BOT,8.0,9.840000


## One more feature
## Season Average

In [56]:
multi_race_df["SeasonAverage"] = (
    multi_race_df.groupby("Abbreviation")["Position"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)
# expanding is used to calculate the average position of the driver for the entire season up to that point

In [57]:
multi_race_df = multi_race_df.dropna(subset=["SeasonAverage"])

In [58]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average", 
    "SeasonAverage"
]].copy()

In [59]:
df = pd.get_dummies(df, columns=["TeamName"])

X = df.drop(columns=["Position"])
y = df["Position"]


In [60]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.0745098039215684


We made a new feature called season average.

We did the  same procedures and turns out this did not make the model better because it caused the Mean Absolute Error to increase from 1.9 to 2.01. 

So this does not mean that the more features we add the better it gets.

So we can choose, either to keep that feature in the long term or we can remove it. But we can experiment and find out

In [61]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average"
]].copy()

In [63]:
multi_race_df = multi_race_df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

In [64]:
multi_race_df["TeamAverage"] = (
    multi_race_df.groupby("TeamName")["Position"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [66]:
multi_race_df[
    multi_race_df["TeamName"] == "Red Bull Racing"
][["RoundNumber", "Position", "TeamAverage"]].head(10)

,RoundNumber,Position,TeamAverage
21,1,2.0,5.558824
80,4,1.0,2.000000
101,5,2.0,1.500000
143,7,4.0,1.666667
165,8,6.0,2.250000
182,9,3.0,3.000000
205,10,6.0,3.000000
222,11,3.0,3.428571
241,12,2.0,3.375000
263,13,4.0,3.222222


In [67]:
multi_race_df = multi_race_df.dropna(subset=["TeamAverage"])

In [76]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average", 
    "TeamAverage"
]].copy()

In [77]:
df = pd.get_dummies(df, columns=["TeamName"])


In [78]:
X = df.drop(columns=["Position"])
y = df["Position"]

In [79]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.322352941176471


Even when we decided to calculate the team average the error still rose.

So this means team average does not help as much for this current state so our best option for this is to DROP TEAM AVERAGE

In [80]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average"
]].copy()

In [ ]:
multi_race_df["Form_vs_Grid"] = (
    multi_race_df["Rolling3Average"] - multi_race_df["GridPosition"]
)
# this is usefule because
# it gives us an idea of how well the driver is performing compared to their starting position

In [82]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average",
    "Form_vs_Grid"
]].copy()

In [83]:
df = pd.get_dummies(df, columns=["TeamName"])

In [84]:
X = df.drop(columns=["Position"])
y = df["Position"]

In [85]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.5976013071895423


Our model is becoming bad little by little and this is because of the test split thing

Our model is basically trained on future races while being tested on the past race, so this breaks the overall logic

So this causes a *DATA LEAKAGE* so we can change it up now

In [86]:
multi_race_df = multi_race_df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

In [87]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average", 
]]

In [88]:
df = pd.get_dummies(df, columns=["TeamName"])

In [89]:
X = df.drop(columns=["Position"])
y = df["Position"]

In [92]:
# Now comes the important step to make our model work properly

split_point = int(0.8 * len(df))
X_train = X.iloc[:split_point]
y_train = y.iloc[:split_point]

X_test = X.iloc[split_point:]
y_test = y.iloc[split_point:]

# So this is important
# Because split point will split 80% of the data for training
# While the remaining 20% will basically be used for the testing process
# So this is realistic because we are trying to predict the future races

In [93]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.5371339869281044


## Since we now put a realistic timeframe for the data to be tested, now we try with the form and grid position feature

In [94]:
multi_race_df = multi_race_df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

In [95]:
multi_race_df["Form_vs_Grid"] = (
    multi_race_df["Rolling3Average"] - multi_race_df["GridPosition"]
)

In [96]:
df = multi_race_df[[
    "TeamName", 
    "GridPosition", 
    "Position", 
    "PreviousPosition", 
    "Rolling3Average", 
    "Form_vs_Grid"
]]

In [97]:
df = pd.get_dummies(df, columns=["TeamName"])

In [98]:
X = df.drop(columns=["Position"])
y = df["Position"]

In [99]:
split_point = int(0.8 * len(df))
X_train = X.iloc[:split_point]
y_train = y.iloc[:split_point]

X_test = X.iloc[split_point:]
y_test = y.iloc[split_point:]

In [100]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [101]:
mean_error = mean_absolute_error(y_test, predictions)
print("Mean Absolute Error:", mean_error)

Mean Absolute Error: 2.660787581699347
